## Road Surface Classification with ViT
This notebook trains a ViT model on the road surface dataset, using the same sampling, seed, and restartable training as the ResNet notebook.

In [ ]:
import os, sys, json, random, numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from src.models.vit_transformer import get_vit
from src.datasets.road_loader import sample_road_surface_dataset

## Set Random Seed

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## Sample and Load Dataset

In [ ]:
base_data = Path('../data/road_surface/RSCD dataset-1million')
sampled_data = Path('../data/road_surface/RSCD_sampled')
if not sampled_data.exists():
    print('Creating sampled dataset...')
    sample_road_surface_dataset(base_data, sampled_data, train_per_class=160, val_per_class=3, test_per_class=10)
    print('Sampling complete!')
else:
    print(f'Sampled dataset already exists at {sampled_data}')

In [ ]:
# Again, defining the transformations, same as the other road_surface notebook 
from torchvision import transforms, datasets
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=sampled_data/'train', transform=train_transform)
val_dataset = datasets.ImageFolder(root=sampled_data/'val', transform=val_transform)
test_dataset = datasets.ImageFolder(root=sampled_data/'test', transform=val_transform)
class_names = train_dataset.classes
num_classes = len(class_names)
print(f'Number of classes: {num_classes}')
print(f'Train samples: {len(train_dataset)}')
print(f'Val samples: {len(val_dataset)}')
print(f'Test samples: {len(test_dataset)}')

In [ ]:
from torch.utils.data import DataLoader
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

## Model Setup

In [ ]:
# Not using pre-trained, loaded via get_vit function, also using M4 gpu acceleration if avail.
model = get_vit(num_classes, pretrained=False)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model.to(device)
# Same loss and optimizer as other notebook
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

## Restartable Training with History

In [ ]:
%matplotlib inline
# Same training loop as other notebook

history_path = 'vit_training_history.json'
# Load previous history if exists
if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)
    train_losses = history.get('train_losses', [])
    val_losses = history.get('val_losses', [])
    train_accs = history.get('train_accs', [])
    val_accs = history.get('val_accs', [])
    start_epoch = len(train_losses)
    print(f'Resuming from epoch {start_epoch}')
    # Set best_val_acc to max historical value if available
    best_val_acc = max(val_accs) if val_accs else 0.0
else:
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    start_epoch = 0
    best_val_acc = 0.0
num_epochs = 100
for epoch in range(start_epoch, num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc='Training', unit='batch'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / total
    train_acc = correct / total * 100
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Validation', unit='batch'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total * 100
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step(val_loss)
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_vit_model.pth')
        print(f'Saved best model (val_acc: {val_acc:.2f}%)')
    # Save history after each epoch
    history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs
    }
    with open(history_path, 'w') as f:
        json.dump(history, f)
    # Plot after each epoch
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, 'b-o', label='Train Loss')
    plt.plot(val_losses, 'r-s', label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, 'b-o', label='Train Acc')
    plt.plot(val_accs, 'r-s', label='Val Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Test Evaluation and Confusion Matrix

In [ ]:
# Plot full training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, color='hotpink', marker='o', label='Train Loss', markersize=6)
plt.plot(val_losses, color='limegreen', marker='o', label='Val Loss', markersize=6)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()
plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(train_accs, color='hotpink', marker='o', label='Train Acc', markersize=6)
plt.plot(val_accs, color='limegreen', marker='o', label='Val Acc', markersize=6)
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('vit_training_history.png', dpi=150)
plt.show()

In [ ]:
# Compute confusion matrix
model.load_state_dict(torch.load('best_vit_model.pth'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing', unit='batch'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
plt.title('Confusion Matrix (Test Set)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()
class_acc = cm.diagonal() / cm.sum(axis=1)
print(f'Mean class accuracy: {np.nanmean(class_acc):.4f}')